# 🏭 Clasificación de Piezas Industriales - Google Colab
## Paso 1: Exploración de Datos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Objetivo:** Explorar el dataset de piezas industriales

---

### ⚙️ Configuración Inicial

In [ ]:
# Verificar si estamos en Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('✅ Ejecutando en Google Colab')
    print('🚀 GPU disponible:', 'SÍ' if __import__('torch').cuda.is_available() else 'NO')
else:
    print('⚠️ No estás en Colab')

### 📁 Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print('\n✅ Google Drive montado correctamente')

### 📚 Importar Librerías

In [ ]:
# Librerías básicas
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import json

# Librerías para imágenes
from PIL import Image
import cv2

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

# Desactivar warnings
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías importadas correctamente')

### 📂 Configurar Rutas

**IMPORTANTE:** Ajusta esta ruta según donde hayas subido tu carpeta en Google Drive

In [ ]:
# 🔧 AJUSTA ESTA RUTA según tu estructura en Google Drive
BASE_DIR = Path('/content/drive/MyDrive/Maestria/tallerML')
DATA_DIR = BASE_DIR / 'datos' / 'DataSet'

# Rutas de los datasets
TRAIN_DIR = DATA_DIR / 'Train_Dataset'
VALID_DIR = DATA_DIR / 'Valid_Dataset'
TEST_DIR = DATA_DIR / 'Test_Dataset'

# Verificar que existen
print(f'📁 Directorio base: {BASE_DIR}')
print(f'📁 Directorio de datos: {DATA_DIR}')
print()
print(f'Existe TRAIN: {TRAIN_DIR.exists()} ✅' if TRAIN_DIR.exists() else f'Existe TRAIN: {TRAIN_DIR.exists()} ❌')
print(f'Existe VALID: {VALID_DIR.exists()} ✅' if VALID_DIR.exists() else f'Existe VALID: {VALID_DIR.exists()} ❌')
print(f'Existe TEST: {TEST_DIR.exists()} ✅' if TEST_DIR.exists() else f'Existe TEST: {TEST_DIR.exists()} ❌')

if not TRAIN_DIR.exists():
    print('\n⚠️ ERROR: No se encontraron los datos. Verifica la ruta en BASE_DIR')

### 🔍 Explorar Estructura del Dataset

In [ ]:
# Listar contenido de Train_Dataset
print('📋 Contenido de Train_Dataset:')
for item in TRAIN_DIR.iterdir():
    if item.is_dir():
        num_files = len(list(item.iterdir()))
        print(f'  📁 {item.name}/ ({num_files} archivos)')
    else:
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f'  📄 {item.name} ({size_mb:.2f} MB)')

### 🏷️ Analizar Archivo de Etiquetas

In [ ]:
# Leer el archivo labels.csv
labels_path = TRAIN_DIR / 'labels.csv'

if labels_path.exists():
    # Leer primeras filas
    df_labels = pd.read_csv(labels_path, nrows=10)
    print('🏷️ Primeras 10 filas del archivo labels.csv:')
    display(df_labels)
    
    print('\n📊 Columnas disponibles:')
    print(df_labels.columns.tolist())
else:
    print('❌ No se encontró el archivo labels.csv')

In [ ]:
# Cargar dataset completo
print('⏳ Cargando dataset completo...')
df_labels_full = pd.read_csv(TRAIN_DIR / 'labels.csv')

print(f'\n📊 Total de imágenes en entrenamiento: {len(df_labels_full):,}')
print(f'\n📋 Información del dataset:')
df_labels_full.info()

### 📈 Análisis de Categorías

In [ ]:
# Identificar columna de categorías
category_col = [col for col in df_labels_full.columns 
                if 'categ' in col.lower() or 'label' in col.lower() or 'class' in col.lower()]

if category_col:
    category_col = category_col[0]
    print(f'✅ Columna de categorías: {category_col}')
else:
    print('⚠️ Columnas disponibles:')
    print(df_labels_full.columns.tolist())
    category_col = df_labels_full.columns[-1]
    print(f'\n➡️ Usando última columna: {category_col}')

In [ ]:
# Contar imágenes por categoría
category_counts = df_labels_full[category_col].value_counts()

print(f'🏭 Número de categorías: {len(category_counts)}')
print(f'\n📊 Distribución de imágenes por categoría:\n')
print(category_counts)

In [ ]:
# Visualizar distribución
plt.figure(figsize=(14, 6))

ax = category_counts.plot(kind='bar', color='steelblue', edgecolor='black', alpha=0.8)
plt.title('Distribución de Imágenes por Categoría (Training Set)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Categoría', fontsize=12)
plt.ylabel('Número de Imágenes', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Valores encima de barras
for i, v in enumerate(category_counts):
    ax.text(i, v + 10, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Análisis de balance
print('\n⚖️ Análisis de Balance de Clases:')
print(f'  • Clase más frecuente: {category_counts.index[0]} ({category_counts.iloc[0]} imágenes)')
print(f'  • Clase menos frecuente: {category_counts.index[-1]} ({category_counts.iloc[-1]} imágenes)')
print(f'  • Ratio de desbalance: {category_counts.iloc[0] / category_counts.iloc[-1]:.2f}x')

if category_counts.iloc[0] / category_counts.iloc[-1] > 2:
    print('\n⚠️ Dataset DESBALANCEADO - Recomendaciones:')
    print('   1. Usar Data Augmentation')
    print('   2. Aplicar class weights en el entrenamiento')
    print('   3. Considerar técnicas de balanceo (SMOTE, etc.)')

### 🖼️ Análisis de Imágenes

In [ ]:
# Ruta de imágenes
images_dir = TRAIN_DIR / 'images'

# Obtener lista de imágenes
image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')) + list(images_dir.glob('*.jpeg'))

print(f'🖼️ Total de archivos de imágenes: {len(image_files):,}')

if len(image_files) == 0:
    print('\n❌ No se encontraron imágenes. Verifica la ruta.')

In [ ]:
# Analizar dimensiones de muestra
sample_size = min(500, len(image_files))
sample_images = np.random.choice(image_files, size=sample_size, replace=False)

image_dimensions = []

print(f'📏 Analizando dimensiones de {sample_size} imágenes aleatorias...')

for img_path in sample_images:
    try:
        img = Image.open(img_path)
        image_dimensions.append(img.size)
    except Exception as e:
        pass

# Convertir a DataFrame
df_dims = pd.DataFrame(image_dimensions, columns=['width', 'height'])

print('\n📊 Estadísticas de dimensiones:')
display(df_dims.describe())

In [ ]:
# Visualizar distribución de dimensiones
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de anchos
axes[0].hist(df_dims['width'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(df_dims['width'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df_dims["width"].mean():.0f}')
axes[0].set_title('Distribución de Anchos', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Ancho (píxeles)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Histograma de alturas
axes[1].hist(df_dims['height'], bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1].axvline(df_dims['height'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df_dims["height"].mean():.0f}')
axes[1].set_title('Distribución de Alturas', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Alto (píxeles)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📐 Dimensiones más comunes:')
print(df_dims.value_counts().head(5))

### 🎨 Visualizar Muestras de Cada Categoría

In [ ]:
# Función para visualizar muestras
def plot_sample_images(df, category_col, images_dir, n_categories=5, n_images=4):
    categories = df[category_col].unique()[:n_categories]
    
    fig, axes = plt.subplots(n_categories, n_images, figsize=(16, 4*n_categories))
    
    for i, category in enumerate(categories):
        category_images = df[df[category_col] == category]
        sample = category_images.sample(min(n_images, len(category_images)))
        
        for j, (idx, row) in enumerate(sample.iterrows()):
            img_filename = row[0] if 'filename' not in df.columns else row['filename']
            img_path = images_dir / str(img_filename)
            
            if img_path.exists():
                img = Image.open(img_path)
                axes[i, j].imshow(img)
                axes[i, j].axis('off')
                
                if j == 0:
                    axes[i, j].set_title(f'{category}\n{img.size[0]}x{img.size[1]}', 
                                        fontsize=12, fontweight='bold')
                else:
                    axes[i, j].set_title(f'{img.size[0]}x{img.size[1]}', fontsize=10)
    
    plt.suptitle('Muestras de Imágenes por Categoría', fontsize=18, fontweight='bold', y=1.0)
    plt.tight_layout()
    plt.show()

# Visualizar muestras
plot_sample_images(df_labels_full, category_col, images_dir, n_categories=5, n_images=4)

### 📋 Resumen Final

In [ ]:
print('='*70)
print('📊 RESUMEN DE EXPLORACIÓN DE DATOS')
print('='*70)
print(f'\n✅ Dataset de Entrenamiento:')
print(f'   • Total de imágenes: {len(df_labels_full):,}')
print(f'   • Número de categorías: {len(category_counts)}')
print(f'   • Dimensiones promedio: {int(df_dims["width"].mean())}x{int(df_dims["height"].mean())} px')
print(f'\n⚖️ Balance de clases:')
print(f'   • Clase más frecuente: {category_counts.iloc[0]} imágenes')
print(f'   • Clase menos frecuente: {category_counts.iloc[-1]} imágenes')
print(f'   • Ratio: {category_counts.iloc[0] / category_counts.iloc[-1]:.2f}x')
print(f'\n🎯 Recomendaciones para el modelo CNN (VGG16):')
print(f'   1. 📏 Redimensionar todas las imágenes a 224x224 (input de VGG16)')
print(f'   2. 🔄 Aplicar Data Augmentation (rotación, flip, zoom)')
print(f'   3. 🧠 Usar Transfer Learning con pesos de ImageNet')
print(f'   4. ⚖️ Aplicar class_weight para compensar desbalance')
print(f'   5. 📊 Normalizar valores de píxeles (0-1 o -1 a 1)')
print('='*70)

### 💾 Guardar Resultados

In [ ]:
# Crear carpeta de resultados
results_dir = BASE_DIR / 'resultados'
results_dir.mkdir(exist_ok=True)

# Guardar estadísticas
stats = {
    'total_images': int(len(df_labels_full)),
    'num_categories': int(len(category_counts)),
    'categories': category_counts.index.tolist(),
    'category_distribution': {k: int(v) for k, v in category_counts.items()},
    'avg_width': int(df_dims['width'].mean()),
    'avg_height': int(df_dims['height'].mean()),
    'min_width': int(df_dims['width'].min()),
    'max_width': int(df_dims['width'].max()),
    'min_height': int(df_dims['height'].min()),
    'max_height': int(df_dims['height'].max())
}

# Guardar JSON
with open(results_dir / 'exploracion_datos.json', 'w') as f:
    json.dump(stats, f, indent=4)

print('✅ Resultados guardados en:', results_dir)
print('\n📄 Archivos generados:')
print(f'   • exploracion_datos.json')

---
## 🎯 Siguiente Paso

**Notebook 02:** Desarrollo del Modelo CNN con Transfer Learning (VGG16)

Ahora que conocemos perfectamente nuestros datos, estamos listos para construir el modelo de clasificación usando VGG16 preentrenado.